<a href="https://colab.research.google.com/github/kartik815/Amazon-ML-Challenge-2026/blob/main/notebooks/03_Normalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive

drive.mount('/content/drive')

import os
import re
import unicodedata
import pandas as pd

DRIVE_ROOT = "/content/drive/MyDrive/Amazon ML Challenge 2026"

DATASET_ROOT = os.path.join(
    DRIVE_ROOT,
    "01_Dataset",
    "6ab10eb3b23ba_student_resource",
    "student_resource",
    "dataset"
)

TRAIN_ROOT = os.path.join(DATASET_ROOT, "train")
TEST_ROOT = os.path.join(DATASET_ROOT, "test")

print("Train path:", TRAIN_ROOT)
print("Test path:", TEST_ROOT)

Mounted at /content/drive
Train path: /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/6ab10eb3b23ba_student_resource/student_resource/dataset/train
Test path: /content/drive/MyDrive/Amazon ML Challenge 2026/01_Dataset/6ab10eb3b23ba_student_resource/student_resource/dataset/test


In [3]:
train_s1 = pd.read_csv(
    os.path.join(TRAIN_ROOT, "train_source1.tsv"),
    sep="\t"
)

train_s2 = pd.read_csv(
    os.path.join(TRAIN_ROOT, "train_source2.tsv"),
    sep="\t"
)

train_s3 = pd.read_csv(
    os.path.join(TRAIN_ROOT, "train_source3.tsv"),
    sep="\t"
)

test_s1 = pd.read_csv(
    os.path.join(TEST_ROOT, "test_source1.tsv"),
    sep="\t"
)

test_s2 = pd.read_csv(
    os.path.join(TEST_ROOT, "test_source2.tsv"),
    sep="\t"
)

test_s3 = pd.read_csv(
    os.path.join(TEST_ROOT, "test_source3.tsv"),
    sep="\t"
)

print("All source datasets loaded.")

All source datasets loaded.


In [4]:
def normalize_basic(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # Lowercase
    text = text.lower()

    # Replace common separators/symbols with spaces
    text = re.sub(r"[/\\|,_;:]+", " ", text)

    # Replace punctuation with spaces
    text = re.sub(r"[^\w\s]", " ", text)

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [5]:
LEGAL_SUFFIX_MAP = {
    "limited": "ltd",
    "ltd": "ltd",

    "llc": "llc",

    "incorporated": "inc",
    "inc": "inc",

    "corporation": "corp",
    "corp": "corp",

    "private": "pvt",
    "pvt": "pvt",

    "company": "co",
    "co": "co",

    "llp": "llp",
    "plc": "plc"
}

In [6]:
def normalize_business_name(text):
    text = normalize_basic(text)

    if not text:
        return ""

    tokens = text.split()

    normalized_tokens = []

    for token in tokens:
        normalized_tokens.append(
            LEGAL_SUFFIX_MAP.get(token, token)
        )

    return " ".join(normalized_tokens)

In [7]:
ADDRESS_ABBREVIATIONS = {
    "street": "st",
    "st": "st",

    "road": "rd",
    "rd": "rd",

    "avenue": "ave",
    "ave": "ave",

    "boulevard": "blvd",
    "blvd": "blvd",

    "drive": "dr",
    "dr": "dr",

    "lane": "ln",
    "ln": "ln",

    "parkway": "pkwy",
    "pkwy": "pkwy",

    "highway": "hwy",
    "hwy": "hwy",

    "place": "pl",
    "pl": "pl",

    "court": "ct",
    "ct": "ct",

    "square": "sq",
    "sq": "sq",

    "apartment": "apt",
    "apt": "apt",

    "suite": "ste",
    "ste": "ste",

    "building": "bldg",
    "bldg": "bldg"
}

In [8]:
def normalize_address(text):
    text = normalize_basic(text)

    if not text:
        return ""

    tokens = text.split()

    normalized_tokens = []

    for token in tokens:
        normalized_tokens.append(
            ADDRESS_ABBREVIATIONS.get(token, token)
        )

    return " ".join(normalized_tokens)

In [9]:
def tokenize_text(text):
    if not text:
        return []

    return text.split()

In [10]:
def add_normalized_columns(df):
    df["name_norm"] = df["business_name"].apply(
        normalize_business_name
    )

    df["address_norm"] = df["business_address"].apply(
        normalize_address
    )

    df["name_tokens"] = df["name_norm"].apply(tokenize_text)

    df["address_tokens"] = df["address_norm"].apply(tokenize_text)

    return df

In [11]:
def add_quality_features(df):

    df["name_missing"] = (
        df["name_norm"].str.len() == 0
    ).astype("int8")

    df["address_missing"] = (
        df["address_norm"].str.len() == 0
    ).astype("int8")

    df["name_token_count"] = (
        df["name_norm"].str.count(r"\S+")
    ).astype("int16")

    df["address_token_count"] = (
        df["address_norm"].str.count(r"\S+")
    ).astype("int16")

    return df

In [12]:
source_datasets = {
    "train_s1": train_s1,
    "train_s2": train_s2,
    "train_s3": train_s3,
    "test_s1": test_s1,
    "test_s2": test_s2,
    "test_s3": test_s3
}

for name, df in source_datasets.items():

    print("Normalizing:", name)

    df["name_norm"] = df["business_name"].apply(
        normalize_business_name
    )

    df["address_norm"] = df["business_address"].apply(
        normalize_address
    )

    df["name_missing"] = (
        df["name_norm"].str.len() == 0
    ).astype("int8")

    df["address_missing"] = (
        df["address_norm"].str.len() == 0
    ).astype("int8")

    df["name_token_count"] = (
        df["name_norm"].str.count(r"\S+")
    ).astype("int16")

    df["address_token_count"] = (
        df["address_norm"].str.count(r"\S+")
    ).astype("int16")

print("Normalization complete.")

Normalizing: train_s1
Normalizing: train_s2
Normalizing: train_s3
Normalizing: test_s1
Normalizing: test_s2
Normalizing: test_s3
Normalization complete.
